# 2. Train and track Logistic Regression

Train one approach and record its result in the same MLflow experiment. Airflow monitors this notebook as its own task.

## Execution parameters
Airflow injects these values into the tagged parameter cell. For manual execution, keep the defaults and run the five notebooks in order.

In [ ]:
# Airflow supplies these values. Defaults also let you run the notebooks manually.
project_root = ""
run_dir = ""
workflow_run_id = ""
experiment_name = "iris-pipeline-comparison"
registered_model_name = "iris-pipeline-classifier"

## Locate files and connect to MLflow
SQLite holds experiment and registry records. This run folder holds data and small result files shared between notebook tasks.

In [ ]:
import json
from pathlib import Path

import mlflow
import pandas as pd
from mlflow import MlflowClient

# Find the project even when VS Code starts in a notebook subfolder.
project_path = Path(project_root).resolve() if project_root else Path.cwd().resolve()
while not (project_path / "requirements.txt").is_file():
    if project_path == project_path.parent:
        raise RuntimeError("Open the notebook from inside the workshop project.")
    project_path = project_path.parent

# Airflow gives each execution its own folder. Manual runs use outputs/manual.
run_path = Path(run_dir) if run_dir else project_path / "outputs" / "manual"
run_path.mkdir(parents=True, exist_ok=True)

# All five notebooks use the same tracking database and registry.
tracking_uri = "sqlite:///" + (project_path / "mlflow.db").as_posix()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
client = MlflowClient()
print("Run folder:", run_path)

## Read the shared split
Each notebook has its own Python session, so it reads the data saved by the first notebook.

In [ ]:
split = json.loads((run_path / "split.json").read_text(encoding="utf-8"))
X_train = pd.DataFrame(split["X_train"], columns=split["feature_names"], dtype="float64")
X_test = pd.DataFrame(split["X_test"], columns=split["feature_names"], dtype="float64")
y_train = split["y_train"]
y_test = split["y_test"]
mlflow.set_experiment(experiment_name)

## Train and log progress
We train logistic regression using SGDClassifier(loss="log_loss"). Each partial_fit call performs one epoch, a pass over the training rows. We use 30 epochs. The simple scaler is fitted only on training data and saved with the classifier.

After every checkpoint, calculate training/test accuracy and log loss and send them to MLflow with a step number. Print the same measurements for Airflow task logs. Training time accumulates fitting time only.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, log_loss, ConfusionMatrixDisplay
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# log_loss makes this an SGD-trained logistic regression model.
# Scale using training rows only; save the scaler with the model for reloading.
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
classifier = SGDClassifier(
    loss="log_loss", learning_rate="constant", eta0=0.02,
    random_state=split["random_state"],
)
model = make_pipeline(scaler, classifier)
epochs = 30
training_seconds = 0.0
with mlflow.start_run(run_name="logistic-regression") as run:
    mlflow.log_params({
        "model_type": "LogisticRegressionSGD", "estimator": "SGDClassifier",
        "loss": classifier.loss, "epochs": epochs,
        "learning_rate": classifier.learning_rate, "eta0": classifier.eta0,
        "alpha": classifier.alpha, "scaler": "StandardScaler",
        "random_state": split["random_state"],
        "test_size": split["test_size"], "dataset": "iris",
    })
    mlflow.set_tag("workflow_run_id", split["execution_id"])
    mlflow.set_tag("progress_unit", "epoch")

    for step in range(1, epochs + 1):
        # One partial_fit call makes one pass over the training rows.
        started = perf_counter()
        classifier.partial_fit(X_train_scaled, y_train, classes=[0, 1, 2])
        training_seconds += perf_counter() - started
        # Evaluate the model at this training checkpoint.
        train_predictions = model.predict(X_train)
        predictions = model.predict(X_test)
        train_accuracy = accuracy_score(y_train, train_predictions)
        test_accuracy = accuracy_score(y_test, predictions)
        train_log_loss = log_loss(y_train, model.predict_proba(X_train), labels=model.classes_)
        test_log_loss = log_loss(y_test, model.predict_proba(X_test), labels=model.classes_)

        # Send numeric history immediately, before the next training step.
        metrics = {
            "train_accuracy": train_accuracy,
            "test_accuracy": test_accuracy,
            "train_log_loss": train_log_loss,
            "test_log_loss": test_log_loss,
            "training_seconds": training_seconds,
            "accuracy": test_accuracy,
        }
        mlflow.log_metrics(metrics, step=step, synchronous=True)
        print(
            f"Step {step}: train_loss={train_log_loss:.4f}, test_loss={test_log_loss:.4f}, "
            f"train_accuracy={train_accuracy:.3f}, test_accuracy={test_accuracy:.3f}",
            flush=True,
        )

    # Keep final scores for comparison; earlier numeric points remain in MLflow.
    accuracy = test_accuracy
    mlflow.log_dict({
        **metrics,
        "final_step": step,
        "train_rows": len(y_train),
        "test_rows": len(y_test),
    }, "evaluation.json")

    # Save an evaluation plot: rows are actual classes, columns are predictions.
    plot = ConfusionMatrixDisplay.from_predictions(
        y_test, predictions,
        labels=model.classes_,
        display_labels=load_iris().target_names,
        cmap="Blues", colorbar=False,
    )
    plot.ax_.set_title("Logistic Regression - test predictions")
    plot.figure_.tight_layout()
    mlflow.log_figure(plot.figure_, "confusion_matrix.png")
    plt.show()
    plt.close(plot.figure_)

    # The signature describes inputs/outputs; the example makes the model inspectable.
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=infer_signature(X_test, predictions),
        input_example=X_train.head(3),
    )

print("Run ID:", run.info.run_id)
print("Training accuracy:", train_accuracy)
print("Test accuracy:", test_accuracy)
print("Training log loss:", train_log_loss)
print("Test log loss:", test_log_loss)
print("Training seconds:", training_seconds)

## Save the handoff
The comparison notebook needs the run ID and saved model location. A small JSON file carries those references.

In [ ]:
# Save references, rather than passing a model object between notebooks.
result = {
    "execution_id": split["execution_id"],
    "model_type": "LogisticRegressionSGD",
    "run_id": run.info.run_id,
    "model_uri": model_info.model_uri,
}
(run_path / "logistic_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
print("Saved result:", run_path / "logistic_result.json")

## Inspect progress in MLflow
Open Metrics and select train_log_loss and test_log_loss to view their histories against Step. Repeat for train_accuracy and test_accuracy. Logistic Regression steps are epochs; Random Forest steps are tree counts. The latest values are used for model comparison.

These histories are numeric MLflow records, not images. Artifacts contains the final confusion_matrix.png and evaluation.json. Airflow task logs contain a printed line per checkpoint.

Iris trains quickly, so the run may finish before you refresh the UI. Its full history remains available afterward. We do not tune settings using the repeatedly measured test scores; these checkpoints demonstrate tracking.